<a href="https://colab.research.google.com/github/yuhui-0611/ESAA/blob/main/OB_0313_%EC%84%B8%EC%85%98_%EC%95%99%EC%83%81%EB%B8%94_%EC%97%B0%EC%8A%B5%EB%AC%B8%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **앙상블 학습과 랜덤 포레스트 연습 문제**
___
- 출처 : 핸즈온 머신러닝 Ch07 앙상블 학습과 랜덤 포레스트 연습문제 2, 7, 8, 9번
- 이론적 지식을 묻는 문제의 경우 텍스트 셀을 추가하여 정답을 적어주세요.

In [1]:
# import libraries
import numpy as np

### **1. 직접 투표와 간접 투표 분류기 사이의 차이점은 무엇일까요?**
___


- 직접 투표
  - 각 분류기의 예측을 모아서 가장 많이 선택된 클래스를 예측하는 것 (= 다수결)
    - 클래스 예측은 다양한 분류기의 예측을 합쳐 최종 클래스를 만든 것으로, 모델 선택의 개념이 아님
    - 모델들이 예측한 클래스 중 가장 많은 것을 최종 예측으로 선택하는 것
  - 다수결 투표로 선택된 분류기가 앙상블에 포함된 개별 분류기 중 가장 뛰어난 것보다도 정확도가 높은 경우가 많음
- 간접 투표
  - 모든 분류기가 클래스의 확률을 예측할 수 있으면(즉, predict_proba() 메서드가 있으면) 개별 분류기의 예측을 평균 내어 확률이 가장 높은 클래스를 예측할 수 있음
  - **확률이 높은 투표에 비중을 더 두기에** 직접 투표보다 성능이 good
  - voting='hard'를 voting='soft'로 바꾸고 모든 분류기가 클래스의 확률을 추정할 수 있으면 됨

### **2. 그레디언트 부스팅 앙상블이 훈련 데이터에 과대 적합되었다면 학습률을 어떻게 해야 할까요?**
___

배깅
- 같은 알고리즘을 사용하고 훈련 세트의 서브셋을 무작위로 구성하여 분류기를 각기 다르게 학습시키는 것
- 훈련 세트에서 중복을 하용하여 샘플링하는 방식
- = 부트스트래핑

부스팅
- 약한 학습기를 여러개 연결하여 강한 학습기를 만드는 앙상블 방법
- 앞의 모델을 보완하며 일련의 예측기를 학습시키는 것
- 종류
  - 에이다부스트 : 이전 모델이 과소적합했던 훈련 샘플의 가중치를 높여 새로운 예측기를 만들어내고, 이 새로운 예측기는 학습하기 어려운 샘플에 더 맞춰지게 됨
  - 그래디언트 부스팅 : 이전 예측기가 만든 잔여 오차에 새로운 예측기를 학습시키며 앙상블에 이전까지의 오차를 보정하도록 예측기를 순차적으로 추가


> Answer : 그래디언트 부스팅 앙상블이 훈련 데이터에 과대 적합되었다면 학습률을 감소시켜야 함




### **3. [실습] 다음 지시에 따라 투표 기반 분류 모델을 만들어 보세요**
___

#### **STEP 1. MNIST 데이터를 불러들이고, 훈련, 검증, 테스트 데이터로 나누세요.**

In [2]:
# import MNIST dataset
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, as_frame = False)
X, y = mnist["data"], mnist["target"]

In [3]:
# train/valid/test dataset
from sklearn.model_selection import train_test_split

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=10000, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=10000, random_state=42)

####  **STEP 2. 랜덤 포레스트 분류기, 엑스트라 트리 분류기, SVM 분류기, MLP 분류기를 훈련시키세요.**
- 모델 파라미터는 `n_estimators=100`, `random_state=42`로 설정합니다.

랜덤 포레스트, 엑스트라 트리 둘다 배깅 기반의 트리 앙상블
- 여러 개의 결정 트리를 만듦
- 각 트리가 독립적으로 예측
- 예측을 모아서 평균 또는 다수결

< 랜덤 포레스트 >
- 각 트리마다 다른 데이터로 학습
- 트리 분할 시, 전체 피처 중 일부만 사용
- 가능한 split을 계산하여 최적 split 선택

< 엑스트라 트리 >
- 전체 데이터 사용
- 랜덤 split 선택
- 랜덤성이 커지면 트리들이 서로 더 달라지고 앙상블 효과가 커지며 과적합이 감소함

In [4]:
# import package
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier

In [5]:
# model fitting

rnd_clf = RandomForestClassifier(n_estimators=100, random_state=42)
ext_clf = ExtraTreesClassifier(n_estimators=100, random_state=42)
svm_clf = LinearSVC(max_iter=100, tol=20, random_state=42)
mlp_clf = MLPClassifier(random_state=42)

rnd_clf.fit(X_train, y_train)
ext_clf.fit(X_train, y_train)
svm_clf.fit(X_train, y_train)
mlp_clf.fit(X_train, y_train)

y_pred_rnd = rnd_clf.predict(X_test)
y_pred_ext = ext_clf.predict(X_test)
y_pred_svm = svm_clf.predict(X_test)
y_pred_mlp = mlp_clf.predict(X_test)

####  **STEP 3-1. 앞에서 훈련시킨 각 모델을 직접 투표 방법을 사용해 앙상블로 연결하고 훈련시킨 후, `score()`메서드를 이용하여 검증 데이터셋에서의 성능을 평가해보세요.**

In [6]:
from sklearn.ensemble import VotingClassifier

In [7]:
# model fitting

voting_clf = VotingClassifier(
    estimators=[('rnd', rnd_clf), ('ext', ext_clf), ('svm', svm_clf), ('mlp', mlp_clf)],
    voting='hard'
)
voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('rnd', RandomForestClassifier(random_state=42)),
                             ('ext', ExtraTreesClassifier(random_state=42)),
                             ('svm',
                              LinearSVC(max_iter=100, random_state=42, tol=20)),
                             ('mlp', MLPClassifier(random_state=42))])

In [8]:
# model test

voting_clf.score(X_val, y_val)

0.9728

####  **STEP 3-2. 검증 데이터셋에서 각 분류 모델의 성능을 `score()` 메서드를 이용하여 확인해보고, 가장 성능이 낮은 모델을 제거하여 그 결과를 비교해보세요.**
- Hint : 가장 성능이 낮은 모델을 제거할 때 `del`를 활용해보세요

In [9]:
# 각 분류 모델 학습

voting_clf.estimators_

[RandomForestClassifier(random_state=42),
 ExtraTreesClassifier(random_state=42),
 LinearSVC(max_iter=100, random_state=42, tol=20),
 MLPClassifier(random_state=42)]

In [10]:
# 각 분류 모델의 성능 확인

estimators = [rnd_clf, ext_clf, svm_clf, mlp_clf]
[estimator.score(X_val, y_val) for estimator in estimators]

[0.9692, 0.9715, 0.0997, 0.9614]

- Q. 어떤 모델의 성능이 가장 낮나요?
- A. SVM

In [11]:
# 가장 성능이 낮은 모델 제거

del voting_clf.estimators_[2]

In [12]:
# model fitting
voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('rnd', RandomForestClassifier(random_state=42)),
                             ('ext', ExtraTreesClassifier(random_state=42)),
                             ('svm',
                              LinearSVC(max_iter=100, random_state=42, tol=20)),
                             ('mlp', MLPClassifier(random_state=42))])

In [13]:
# 모델 제거 후 성능 확인

voting_clf.score(X_val, y_val)

0.9728

- SVM이 틀렸지만 다수결 결과는 변하지 않기 때문에 제거 전과 후의 성능이 같게 나오는 것

### **4. 다음 단계를 따라 앞에서 훈련시킨 분류 모델들을 이용하여 스태킹 앙상블을 구성해보자.**
___

- 여러개의 모델을 학습시킨 뒤 그 모델들의 예측 결과를 다시 하나의 모델이 학습해 최종 예측을 만드는 방법
> 모델들의 예측을 피처로 사용하여 새로운 모델이 최종 판단을 하는 앙상블

| 방법        | 방식               |
| --------- | ---------------- |
| Voting    | 다수결              |
| Averaging | 평균               |
| Stacking  | **새 모델이 예측을 학습** |


#### **STEP 1. 3번 문제의 각 분류 모델을 실행해서 검증 세트에서 예측을 만들고, 그 결과로 훈련 세트를 만들어 보세요.**

In [14]:
# 새 훈련 세트를 저장할 ndarray 생성
X_val_predictions = np.empty((len(X_val), len(estimators)), dtype=np.float32)
X_val_predictions

array([[4.5637139e-10, 4.5253533e-41, 4.0820435e+01, 0.0000000e+00],
       [4.0820435e+01, 0.0000000e+00, 4.0820435e+01, 0.0000000e+00],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00],
       ...,
       [8.4077908e-45, 0.0000000e+00, 9.8090893e-45, 0.0000000e+00],
       [1.4012985e-45, 0.0000000e+00, 1.2611686e-44, 0.0000000e+00],
       [9.8090893e-45, 0.0000000e+00, 9.8090893e-45, 0.0000000e+00]],
      dtype=float32)

In [17]:
# 새 훈련 세트 생성
for index, estimator in enumerate(estimators):
    X_val_predictions[:, index] = estimator.predict(X_val)

####  **STEP 2. 새로운 훈련 세트를 이용하여 랜덤 포레스트 분류 모델을 학습시켜 보세요.**

In [19]:
rnd_forest_blender = RandomForestClassifier(n_estimators=100, random_state=42)
rnd_forest_blender.fit(X_val_predictions, y_val)

RandomForestClassifier(random_state=42)

- 이 랜덤 포레스트 분류 모델이 바로 블렌더에 해당합니다.

####  **STEP 3. 이제 테스트셋에서 스태킹 앙상블 모델을 평가해보세요.**
- 성능 평가 지표로 **정확도**를 이용하세요.

In [20]:
# 각 분류 모델의 예측을 만들어 새로운 데이터셋 생성
X_test_predictions = np.empty((len(X_test), len(estimators)), dtype=np.float32)

for index, estimator in enumerate(estimators):
    X_test_predictions[:, index] = estimator.predict(X_test)

In [21]:
# 새로운 데이터셋을 이용하여 블렌더로 예측
y_pred = rnd_forest_blender.predict(X_test_predictions)

In [22]:
# model test
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

0.9683